# 01 – Chargement et contrôle qualité des données de matchs

**Projet :** NBA Betting Analytics (analyse statistique et simulation uniquement, aucune mise réelle).

**Objectif de ce notebook :** charger la table des matchs, vérifier qu'elle est techniquement fiable
(valeurs manquantes, types, dates, doublons, cohérence métier) et produire quelques statistiques descriptives.

**Données utilisées :** des données **synthétiques** générées par `src/nba_analytics/synthetic.py`.
Ce notebook sert de banc d'essai des contrôles ; les données réelles sont traitées dans `02_donnees_reelles_fivethirtyeight.ipynb`.
Les équipes T01 à T30 sont fictives. Les chiffres ci-dessous décrivent le générateur, pas la NBA.

Ce notebook ne construit **aucun modèle de prédiction**.

## 0. Installation (Colab) ou détection du dépôt (local)

Dans Colab, la cellule clone le dépôt GitHub public, **sans jeton** : aucun secret n'est nécessaire pour lire.

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_USER = "JohnGAB7"
REPO_NAME = "Sport-Analytics"
PROJECT_DIR = "nba"                  # le projet NBA est dans ce sous-dossier du dépôt
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content") / REPO_NAME
    if not repo.exists():
        # Dépôt public : clonage sans jeton (aucun secret dans le notebook ni dans .git/config).
        url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
        subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    repo = repo / PROJECT_DIR
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Exécution locale : on remonte jusqu'au dossier qui contient src/
    here = Path.cwd()
    repo = next(p for p in [here, *here.parents] if (p / "src" / "nba_analytics").exists())
    os.chdir(repo)

sys.path.insert(0, str(repo / "src"))
print("Dossier du dépôt :", Path.cwd().name)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nba_analytics import quality
from nba_analytics.synthetic import generate_games, inject_defects

pd.set_option("display.width", 140)
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Génération et sauvegarde des données synthétiques

On génère 3 saisons régulières (2022, 2023, 2024), avec une graine fixe pour que le résultat soit **reproductible**.
On écrit le résultat en CSV puis on le **relit en texte brut** : c'est la situation réelle d'un fichier externe,
où rien ne garantit les types.

In [ ]:
DATA_PATH = Path("data/synthetic/games_synthetic.csv")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

games = generate_games(seasons=[2022, 2023, 2024], seed=42)
games.to_csv(DATA_PATH, index=False)

raw = quality.load_games_csv(DATA_PATH)   # tout est lu comme du texte
print(raw.shape)
raw.head()

## 2. Contrôles de qualité

In [ ]:
res = quality.run_all_checks(raw)
print("Colonnes obligatoires absentes :", res["missing_columns"] or "aucune")
pd.Series(res["summary"], name="valeur").to_frame()

### 2.1 Valeurs manquantes

In [ ]:
res["missing"]

### 2.2 Types : attendus contre effectifs (après conversion contrôlée)

In [ ]:
print("Valeurs présentes mais non convertibles :", res["conversion_failures"])
res["dtypes"]

### 2.3 Dates : absentes, futures, hors saison, incohérentes avec la saison

In [ ]:
games_ok = res["data"]
print("Période :", games_ok["game_date"].min().date(), "->", games_ok["game_date"].max().date())
print("Lignes avec un problème de date :", len(res["dates"]))
res["dates"].head()

### 2.4 Doublons et incohérences métier

In [ ]:
for name, d in res["duplicates"].items():
    print(f"Doublons [{name}] : {len(d)} lignes")
print("Équipes avec 2 matchs le même jour :", len(res["double_booked"]))
print("Incohérences métier :", len(res["logic"]))

## 3. Les contrôles détectent-ils vraiment les problèmes ?

Un contrôle qui ne signale jamais rien peut être cassé. On injecte donc des défauts **connus**
(scores manquants, date illisible, date en août, doublons, équipe contre elle-même, score négatif, match nul)
et on vérifie qu'ils sont tous retrouvés.

In [ ]:
dirty_path = Path("data/synthetic/games_synthetic_dirty.csv")
inject_defects(games, seed=0).to_csv(dirty_path, index=False)
res_dirty = quality.run_all_checks(quality.load_games_csv(dirty_path))
pd.DataFrame({"propre": res["summary"], "avec défauts": res_dirty["summary"]})

In [ ]:
cols = ["game_id", "game_date", "home_team", "away_team", "home_pts", "away_pts",
        "same_team", "negative_pts", "implausible_pts", "tie"]
res_dirty["logic"][cols]

In [ ]:
res_dirty["dates"][["game_id", "season", "game_date", "date_missing", "off_season_month", "season_mismatch"]]

## 4. Statistiques descriptives (données propres)

Rappel : ces valeurs reflètent les paramètres du générateur (moyenne 112 points, écart-type 12, avantage du terrain +2,5),
pas la réalité de la NBA.

In [ ]:
stats = quality.descriptive_stats(games_ok)
stats

In [ ]:
d = games_ok.assign(margin=(games_ok.home_pts - games_ok.away_pts).astype(float),
                    total=(games_ok.home_pts + games_ok.away_pts).astype(float))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(d["margin"], bins=40, color="#4C72B0")
axes[0].axvline(0, color="black", lw=1)
axes[0].set(title="Écart de points (domicile - extérieur)", xlabel="points", ylabel="matchs")
axes[1].hist(d["total"], bins=40, color="#55A868")
axes[1].set(title="Total de points par match", xlabel="points")
plt.tight_layout(); plt.show()

In [ ]:
# Bilan victoires/défaites par équipe (saison 2024), pour vérifier la dispersion des niveaux.
s24 = d[d.season == 2024]
wins = pd.concat([s24.loc[s24.margin > 0, "home_team"], s24.loc[s24.margin < 0, "away_team"]]).value_counts()
played = pd.concat([s24.home_team, s24.away_team]).value_counts()
table = pd.DataFrame({"matchs": played, "victoires": wins}).fillna(0).astype(int)
table["pct_victoires"] = (table.victoires / table.matchs).round(3)
table.sort_values("pct_victoires", ascending=False).head(10)

## 5. Conclusion de l'étape

- Les contrôles passent sur les données propres et détectent chaque défaut injecté.
- La table synthétique est prête à servir de banc d'essai.
- Les mêmes contrôles sont appliqués aux données réelles dans le notebook 02.

Limites connues : voir `docs/LIMITATIONS.md`.